# PySpark Warm-up

> `spark` and `sc` are available automatically from the kernel startup.

In [1]:
import concurrent.futures
import pyspark
import random
import time

from delta import DeltaTable

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import _parse_datatype_string

from pyspark.sql.window import Window

In [2]:
spark

# Window function

## Top N per group
Given a sales DataFrame with columns (region, salesperson, revenue), find the top 2 salespersons by revenue in each region.

In [3]:
data = [("West","Alice",9200),("West","Bob",8100),("West","Carol",7500),
       ("East","Dave",8800),("East","Eve",9500),("East","Frank",7200)]
df = spark.createDataFrame(data, ["region","salesperson","revenue"])

In [4]:
df.show()

+------+-----------+-------+
|region|salesperson|revenue|
+------+-----------+-------+
|  West|      Alice|   9200|
|  West|        Bob|   8100|
|  West|      Carol|   7500|
|  East|       Dave|   8800|
|  East|        Eve|   9500|
|  East|      Frank|   7200|
+------+-----------+-------+



In [8]:
window = Window.partitionBy("region").orderBy(desc("revenue"))

df = df.withColumn("rank", dense_rank().over(window)).filter("rank <= 2").drop("rank")



In [9]:
df.show()

+------+-----------+-------+
|region|salesperson|revenue|
+------+-----------+-------+
|  East|        Eve|   9500|
|  East|       Dave|   8800|
|  West|      Alice|   9200|
|  West|        Bob|   8100|
+------+-----------+-------+



## Deduplicate keep latest

A user_events table has (user_id, event_type, event_ts). Remove duplicates keeping only the most recent record per user_id + event_type combination.

In [10]:
data = [(1,"login","2024-01-03"),(1,"login","2024-01-05"),
       (2,"purchase","2024-01-02"),(2,"purchase","2024-01-07")]
df = spark.createDataFrame(data, ["user_id","event_type","event_ts"])

In [11]:
df.show()

+-------+----------+----------+
|user_id|event_type|  event_ts|
+-------+----------+----------+
|      1|     login|2024-01-03|
|      1|     login|2024-01-05|
|      2|  purchase|2024-01-02|
|      2|  purchase|2024-01-07|
+-------+----------+----------+



In [15]:
window = Window.partitionBy("user_id", "event_type").orderBy(desc("event_ts"))

df = df.withColumn("rank", row_number().over(window)).filter("rank = 1").drop("rank")

In [16]:
df.show()

+-------+----------+----------+
|user_id|event_type|  event_ts|
+-------+----------+----------+
|      1|     login|2024-01-05|
|      2|  purchase|2024-01-07|
+-------+----------+----------+



## 7-day rolling average

Given daily_sales DataFrame with (sale_date, amount), compute a 7-day rolling average of amount for each date.

rowsBetween vs rangeBetween

In [17]:
from pyspark.sql.functions import to_date
data = [("2024-01-01",100),("2024-01-02",150),("2024-01-03",200),
        ("2024-01-04",130),("2024-01-05",90),("2024-01-06",180),
        ("2024-01-07",160),("2024-01-08",210)]
df = spark.createDataFrame(data,["sale_date","amount"])
df = df.withColumn("sale_date", to_date("sale_date"))

In [22]:
df.show()
df.schema

+----------+------+
| sale_date|amount|
+----------+------+
|2024-01-01|   100|
|2024-01-02|   150|
|2024-01-03|   200|
|2024-01-04|   130|
|2024-01-05|    90|
|2024-01-06|   180|
|2024-01-07|   160|
|2024-01-08|   210|
+----------+------+



StructType([StructField('sale_date', DateType(), True), StructField('amount', LongType(), True)])

In [28]:
window = Window.orderBy(col("sale_date").cast("timestamp").cast("long").asc()).rangeBetween(-6 * 86400, 0)

df = df.withColumn("rolling_7d_avg", round(avg("amount").over(window), 2))

In [29]:
df.show()

+----------+------+--------------+
| sale_date|amount|rolling_7d_avg|
+----------+------+--------------+
|2024-01-01|   100|         100.0|
|2024-01-02|   150|         125.0|
|2024-01-03|   200|         150.0|
|2024-01-04|   130|         145.0|
|2024-01-05|    90|         134.0|
|2024-01-06|   180|        141.67|
|2024-01-07|   160|        144.29|
|2024-01-08|   210|         160.0|
+----------+------+--------------+



26/04/24 01:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 01:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 01:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 01:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 01:41:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Sessionize clickstream

Given clickstream data (user_id, page, timestamp_ms), assign a session_id to each user. A new session starts when the gap between consecutive events exceeds 30 minutes.

In [30]:
data = [(1,"home",0),(1,"cart",5*60000),(1,"checkout",10*60000),
       (1,"home",50*60000),(1,"cart",55*60000),
       (2,"home",0),(2,"product",2*60000)]
df = spark.createDataFrame(data,["user_id","page","ts_ms"])

In [31]:
df.show()

+-------+--------+-------+
|user_id|    page|  ts_ms|
+-------+--------+-------+
|      1|    home|      0|
|      1|    cart| 300000|
|      1|checkout| 600000|
|      1|    home|3000000|
|      1|    cart|3300000|
|      2|    home|      0|
|      2| product| 120000|
+-------+--------+-------+



In [33]:
window = Window.partitionBy("user_id").orderBy(asc("ts_ms"))

df = df.withColumn("prev_ts", lag("ts_ms").over(window))

In [35]:
SESSION_TIMEOUT_MS = 30 * 60_000
df = df.withColumn("is_new_session", when(col("prev_ts").isNull() | (col("ts_ms") - col("prev_ts") > SESSION_TIMEOUT_MS), 1).otherwise(0))

df = df.withColumn("session_num", sum("is_new_session").over(window)).withColumn("session_id", concat_ws("_", col("user_id"), col("session_num")))

In [36]:
df.show()

+-------+--------+-------+-------+--------------+-----------+----------+
|user_id|    page|  ts_ms|prev_ts|is_new_session|session_num|session_id|
+-------+--------+-------+-------+--------------+-----------+----------+
|      1|    home|      0|   NULL|             1|          1|       1_1|
|      1|    cart| 300000|      0|             0|          1|       1_1|
|      1|checkout| 600000| 300000|             0|          1|       1_1|
|      1|    home|3000000| 600000|             1|          2|       1_2|
|      1|    cart|3300000|3000000|             0|          2|       1_2|
|      2|    home|      0|   NULL|             1|          1|       2_1|
|      2| product| 120000|      0|             0|          1|       2_1|
+-------+--------+-------+-------+--------------+-----------+----------+



# Dataframe API

## Flatten nested JSON

You receive JSON with nested structs and arrays. Flatten it into a single-level DataFrame.

* explode: drops empty arrays 
* explode_outer: keeps empty arrays as null
* posexplode: adds index position
* array_join: Array → single string

In [62]:
from pyspark.sql.types import *
data = [('{"user":{"id":1,"name":"Alice"},"tags":["vip","active"]}',),
        ('{"user":{"id":2,"name":"Bob"},"tags":["trial"]}',)]
df = spark.read.json(spark.sparkContext.parallelize([r[0] for r in data]))

In [63]:
df.printSchema()
df.show()

root
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)

+-------------+----------+
|         tags|      user|
+-------------+----------+
|[vip, active]|{1, Alice}|
|      [trial]|  {2, Bob}|
+-------------+----------+



In [64]:
df = df.withColumn("tag", explode(col("tags")))

In [65]:
df.show()

+-------------+----------+------+
|         tags|      user|   tag|
+-------------+----------+------+
|[vip, active]|{1, Alice}|   vip|
|[vip, active]|{1, Alice}|active|
|      [trial]|  {2, Bob}| trial|
+-------------+----------+------+



In [69]:
df.select("user.id", "user.name", "tag").show()

+---+-----+------+
| id| name|   tag|
+---+-----+------+
|  1|Alice|   vip|
|  1|Alice|active|
|  2|  Bob| trial|
+---+-----+------+



## Pivot monthly sales

Transform a (product, month, sales) DataFrame into a wide format with one column per month.

In [79]:
data = [("TV","Jan",300),("TV","Feb",250),("Phone","Jan",500),("Phone","Feb",480)]
df = spark.createDataFrame(data,["product","month","sales"])

In [80]:
df.show()

+-------+-----+-----+
|product|month|sales|
+-------+-----+-----+
|     TV|  Jan|  300|
|     TV|  Feb|  250|
|  Phone|  Jan|  500|
|  Phone|  Feb|  480|
+-------+-----+-----+



In [82]:
df = df.groupBy("product").pivot("month", ["Jan", "Feb"]).sum("sales")

In [83]:
df.show()

+-------+---+---+
|product|Jan|Feb|
+-------+---+---+
|     TV|300|250|
|  Phone|500|480|
+-------+---+---+



## Read, transform, write pipeline

Read a CSV of transactions (txn_id, user_id, amount, status), filter to status='completed', compute total_amount per user, and write as Parquet partitioned by user_id.

In [89]:
# Input CSV cols: txn_id, user_id, amount, status
df = spark.read.option("header",True).option("inferSchema",True).csv("../data/txns.csv")

In [90]:
df.show()

+------+-------+-------+---------+
|txn_id|user_id| amount|   status|
+------+-------+-------+---------+
|TXN001| USR101|  250.0|completed|
|TXN002| USR102|   89.5|  pending|
|TXN003| USR103|1200.75|completed|
|TXN004| USR101|   45.0|   failed|
|TXN005| USR104|  320.0|completed|
|TXN006| USR105|  15.99|  pending|
|TXN007| USR102|  500.0|   failed|
|TXN008| USR106|  78.25|completed|
|TXN009| USR103|  999.0|  pending|
|TXN010| USR107|   60.0|completed|
+------+-------+-------+---------+



In [92]:
df = df.filter("status = 'completed'").groupBy("user_id").agg(sum("amount").alias("total_amount"))

In [93]:
df.show()

+-------+------------+
|user_id|total_amount|
+-------+------------+
| USR103|     1200.75|
| USR101|       250.0|
| USR107|        60.0|
| USR106|       78.25|
| USR104|       320.0|
+-------+------------+



In [96]:
df.coalesce(1).write.partitionBy('user_id').option("header", "true").csv("../data/output/total_txns")

# Analytics

## Compute funnel conversion

Given an events DataFrame (user_id, event, ts), compute funnel conversion: view → add_to_cart → purchase. Count users at each stage.

### Approach 1

In [99]:
data = [(1,"view",1),(1,"add_to_cart",2),(1,"purchase",3),
       (2,"view",1),(2,"add_to_cart",2),
       (3,"view",1),(4,"view",1),(4,"purchase",2)]
df = spark.createDataFrame(data,["user_id","event","ts"])

In [100]:
df.show()

+-------+-----------+---+
|user_id|      event| ts|
+-------+-----------+---+
|      1|       view|  1|
|      1|add_to_cart|  2|
|      1|   purchase|  3|
|      2|       view|  1|
|      2|add_to_cart|  2|
|      3|       view|  1|
|      4|       view|  1|
|      4|   purchase|  2|
+-------+-----------+---+



In [101]:
df = df.groupBy("user_id").pivot("event", ["view", "add_to_cart", "purchase"]).count().fillna(0)

In [102]:
df = df.agg(sum("view").alias("view"), sum("add_to_cart").alias("add_to_cart"), sum("purchase").alias("purchase"))

In [103]:
df.show()

+----+-----------+--------+
|view|add_to_cart|purchase|
+----+-----------+--------+
|   4|          2|       2|
+----+-----------+--------+



### Approach 2

In [104]:
data = [(1,"view",1),(1,"add_to_cart",2),(1,"purchase",3),
       (2,"view",1),(2,"add_to_cart",2),
       (3,"view",1),(4,"view",1),(4,"purchase",2)]
df = spark.createDataFrame(data,["user_id","event","ts"])

In [105]:
df = df.groupBy("user_id").agg(
    max(when(col("event") == "view", 1).otherwise(0)).alias("did_view"),
    max(when(col("event") == "add_to_cart", 1).otherwise(0)).alias("did_add"),
    max(when(col("event") == "purchase", 1).otherwise(0)).alias("did_purchase"),
)

In [106]:
df = df.agg(sum("did_view").alias("view"), sum("did_add").alias("add_to_cart"), sum("did_purchase").alias("purchase"))

In [107]:
df.show()

+----+-----------+--------+
|view|add_to_cart|purchase|
+----+-----------+--------+
|   4|          2|       2|
+----+-----------+--------+



### Approach 3

In [108]:
data = [(1,"view",1),(1,"add_to_cart",2),(1,"purchase",3),
       (2,"view",1),(2,"add_to_cart",2),
       (3,"view",1),(4,"view",1),(4,"purchase",2)]
df = spark.createDataFrame(data,["user_id","event","ts"])

In [109]:
df = df.groupBy("user_id", "event").agg(min("ts").alias("first_ts"))

In [110]:
df = df.groupBy("user_id").pivot(
    "event", ["view", "add_to_cart", "purchase"]
).agg(min("first_ts")).fillna(9999)

In [111]:
df = df.withColumn(
    "valid_view",
    when(col("view") < 9999, 1).otherwise(0)
).withColumn(
    "valid_add",
    when((col("add_to_cart") < 9999) &
         (col("add_to_cart") > col("view")), 1).otherwise(0)
).withColumn(
    "valid_purchase",
    when((col("purchase") < 9999) &
         (col("purchase") > col("add_to_cart")), 1).otherwise(0)
)

In [112]:
df = df.agg(
    sum("valid_view").alias("viewed"),
    sum("valid_add").alias("added_to_cart"),
    sum("valid_purchase").alias("purchased"),
)

In [113]:
df.show()

+------+-------------+---------+
|viewed|added_to_cart|purchased|
+------+-------------+---------+
|     4|            2|        1|
+------+-------------+---------+



### Approach 4 (with repeat conversions)

In [124]:
data = [(1,"view",1),(1,"add_to_cart",2),(1,"purchase",3),
        (1,"view",4),(1,"add_to_cart",5),(1,"purchase",6),
       (2,"view",1),(2,"add_to_cart",2),
       (3,"view",1),(4,"view",1),(4,"purchase",2)]
df = spark.createDataFrame(data,["user_id","event","ts"])

In [125]:
window = Window.partitionBy("user_id").orderBy(asc("ts"))

In [127]:
df = df.withColumn("is_funnel_start", when(expr("event == 'view'"), 1).otherwise(0)).withColumn("attempt", sum("is_funnel_start").over(window))

In [128]:
df.show()

+-------+-----------+---+---------------+-------+
|user_id|      event| ts|is_funnel_start|attempt|
+-------+-----------+---+---------------+-------+
|      1|       view|  1|              1|      1|
|      1|add_to_cart|  2|              0|      1|
|      1|   purchase|  3|              0|      1|
|      1|       view|  4|              1|      2|
|      1|add_to_cart|  5|              0|      2|
|      1|   purchase|  6|              0|      2|
|      2|       view|  1|              1|      1|
|      2|add_to_cart|  2|              0|      1|
|      3|       view|  1|              1|      1|
|      4|       view|  1|              1|      1|
|      4|   purchase|  2|              0|      1|
+-------+-----------+---+---------------+-------+



In [129]:
window = Window.partitionBy("user_id", "attempt").orderBy("ts")

In [130]:
df = df.groupBy("user_id", "attempt", "event").agg(min("ts").alias("step_ts"))

In [131]:
df = df.groupBy("user_id", "attempt").pivot("event", ["view", "add_to_cart", "purchase"]).agg(min("step_ts")).fillna(99999)

In [132]:
df.show()

+-------+-------+----+-----------+--------+
|user_id|attempt|view|add_to_cart|purchase|
+-------+-------+----+-----------+--------+
|      1|      1|   1|          2|       3|
|      1|      2|   4|          5|       6|
|      2|      1|   1|          2|   99999|
|      3|      1|   1|      99999|   99999|
|      4|      1|   1|      99999|       2|
+-------+-------+----+-----------+--------+



In [133]:
df = df.withColumn(
    "valid_view",
    when(col("view") < 9999, 1).otherwise(0)
).withColumn(
    "valid_add",
    when((col("add_to_cart") < 9999) &
         (col("add_to_cart") > col("view")), 1).otherwise(0)
).withColumn(
    "valid_purchase",
    when((col("purchase") < 9999) &
         (col("purchase") > col("add_to_cart")), 1).otherwise(0)
)

In [134]:
df = df.agg(
    sum("valid_view").alias("viewed"),
    sum("valid_add").alias("added_to_cart"),
    sum("valid_purchase").alias("purchased"),
)

In [135]:
df.show()

+------+-------------+---------+
|viewed|added_to_cart|purchased|
+------+-------------+---------+
|     5|            3|        2|
+------+-------------+---------+



# Performance

## Skewed join with salting

You have a massive orders table and a small but skewed products table (one product_id has 80% of the data). Optimize the join.

In [137]:
# orders: (order_id, product_id, qty)  — 500M rows
# products: (product_id, name, price) — skewed on product_id='HOT'
# Normal join causes OOM / slow tasks on 1-2 executors

In [138]:
orders_data = (
    [("HOT", f"order_{i}",  100) for i in range(8)] +   # 8 HOT rows
    [("P002", f"order_{i}", 200) for i in range(8, 9)] + # 1 P002 row
    [("P003", f"order_{i}", 300) for i in range(9, 10)]  # 1 P003 row
)
orders = spark.createDataFrame(orders_data, ["product_id", "order_id", "amount"])

products_data = [
    ("HOT",  "Viral Gadget",  999.99),
    ("P002", "Boring Widget",  19.99),
    ("P003", "Niche Tool",     49.99),
]
products = spark.createDataFrame(products_data, ["product_id", "name", "price"])
orders.join(products, "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#1302, order_id#1303, amount#1304L, name#1306, price#1307]
   +- SortMergeJoin [product_id#1302], [product_id#1305], Inner
      :- Sort [product_id#1302 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id#1302, 200), ENSURE_REQUIREMENTS, [plan_id=2552]
      :     +- Filter isnotnull(product_id#1302)
      :        +- Scan ExistingRDD[product_id#1302,order_id#1303,amount#1304L]
      +- Sort [product_id#1305 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(product_id#1305, 200), ENSURE_REQUIREMENTS, [plan_id=2553]
            +- Filter isnotnull(product_id#1305)
               +- Scan ExistingRDD[product_id#1305,name#1306,price#1307]




In [139]:
orders.join(broadcast(products), "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#1302, order_id#1303, amount#1304L, name#1306, price#1307]
   +- BroadcastHashJoin [product_id#1302], [product_id#1305], Inner, BuildRight, false
      :- Filter isnotnull(product_id#1302)
      :  +- Scan ExistingRDD[product_id#1302,order_id#1303,amount#1304L]
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=2582]
         +- Filter isnotnull(product_id#1305)
            +- Scan ExistingRDD[product_id#1305,name#1306,price#1307]




In [140]:
SALT_BUCKETS = 5   # tune this: more buckets = better spread, more replication cost

# ── Step 1: salt the large (skewed) table ─────────────────────────────────────
# append a random integer 0..N-1 to every product_id
# HOT_0, HOT_1, HOT_2, HOT_3, HOT_4 — spread across 5 partitions

orders_salted = orders.withColumn(
    "salt",
    floor(rand() * SALT_BUCKETS).cast("int")
).withColumn(
    "product_id_salted",
    concat_ws("_", col("product_id"), col("salt"))
)

orders_salted.show()

+----------+--------+------+----+-----------------+
|product_id|order_id|amount|salt|product_id_salted|
+----------+--------+------+----+-----------------+
|       HOT| order_0|   100|   4|            HOT_4|
|       HOT| order_1|   100|   2|            HOT_2|
|       HOT| order_2|   100|   0|            HOT_0|
|       HOT| order_3|   100|   4|            HOT_4|
|       HOT| order_4|   100|   0|            HOT_0|
|       HOT| order_5|   100|   2|            HOT_2|
|       HOT| order_6|   100|   4|            HOT_4|
|       HOT| order_7|   100|   4|            HOT_4|
|      P002| order_8|   200|   4|           P002_4|
|      P003| order_9|   300|   0|           P003_0|
+----------+--------+------+----+-----------------+



In [155]:
# ── Step 2: replicate the small table N times with matching salts ─────────────
# HOT must appear as HOT_0, HOT_1, HOT_2, HOT_3, HOT_4
# so every salted order row still finds its product

# generate array [0, 1, 2, 3, 4] then explode → one row per salt value
salt_array = array([lit(i) for i in range(SALT_BUCKETS)])

# salt_array = sequence(0, SALT_BUCKETS)

products_replicated = products \
    .withColumn("salt", explode(salt_array)) \
    .withColumn(
        "product_id_salted",
        concat_ws("_", col("product_id"), col("salt"))
    )

products_replicated.show()

+----------+-------------+------+----+-----------------+
|product_id|         name| price|salt|product_id_salted|
+----------+-------------+------+----+-----------------+
|       HOT| Viral Gadget|999.99|   0|            HOT_0|
|       HOT| Viral Gadget|999.99|   1|            HOT_1|
|       HOT| Viral Gadget|999.99|   2|            HOT_2|
|       HOT| Viral Gadget|999.99|   3|            HOT_3|
|       HOT| Viral Gadget|999.99|   4|            HOT_4|
|      P002|Boring Widget| 19.99|   0|           P002_0|
|      P002|Boring Widget| 19.99|   1|           P002_1|
|      P002|Boring Widget| 19.99|   2|           P002_2|
|      P002|Boring Widget| 19.99|   3|           P002_3|
|      P002|Boring Widget| 19.99|   4|           P002_4|
|      P003|   Niche Tool| 49.99|   0|           P003_0|
|      P003|   Niche Tool| 49.99|   1|           P003_1|
|      P003|   Niche Tool| 49.99|   2|           P003_2|
|      P003|   Niche Tool| 49.99|   3|           P003_3|
|      P003|   Niche Tool| 49.9

In [150]:
# ── Step 3: join on the salted key ────────────────────────────────────────────
result = orders_salted.join(
    products_replicated,
    on="product_id_salted",
    how="inner"
).drop("salt", "product_id_salted")   # clean up salt columns

result.show()

+----------+--------+------+----------+-------------+------+
|product_id|order_id|amount|product_id|         name| price|
+----------+--------+------+----------+-------------+------+
|       HOT| order_2|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_4|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_1|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_5|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_0|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_3|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_6|   100|       HOT| Viral Gadget|999.99|
|       HOT| order_7|   100|       HOT| Viral Gadget|999.99|
|      P002| order_8|   200|      P002|Boring Widget| 19.99|
|      P003| order_9|   300|      P003|   Niche Tool| 49.99|
+----------+--------+------+----------+-------------+------+



In [151]:
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#1302, order_id#1303, amount#1304L, product_id#1305, name#1306, price#1307]
   +- SortMergeJoin [product_id_salted#1309], [product_id_salted#1347], Inner
      :- Sort [product_id_salted#1309 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id_salted#1309, 200), ENSURE_REQUIREMENTS, [plan_id=2796]
      :     +- Project [product_id#1302, order_id#1303, amount#1304L, concat_ws(_, product_id#1302, cast(salt#1308 as string)) AS product_id_salted#1309]
      :        +- Project [product_id#1302, order_id#1303, amount#1304L, cast(FLOOR((rand(-2660194058780391416) * 5.0)) as int) AS salt#1308]
      :           +- Scan ExistingRDD[product_id#1302,order_id#1303,amount#1304L]
      +- Sort [product_id_salted#1347 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(product_id_salted#1347, 200), ENSURE_REQUIREMENTS, [plan_id=2797]
            +- Project [product_id#1305, name#1306, 

# Iceberg/Delta

## SCD Type 2 merge

Implement SCD Type 2 on a customers table: keep history of address changes with is_current, effective_date, and expiry_date columns.

In [156]:
# Existing dimension table (Delta/Iceberg)
# cols: customer_id, address, is_current, eff_date, exp_date

# Incoming updates
updates = [(1,"456 Elm St","2024-02-01"),(3,"789 Oak Ave","2024-02-01")]
upd_df = spark.createDataFrame(updates,["customer_id","new_address","change_date"])